# Phase 2 Verification — Kiểm tra tích hợp Neo4j + Qdrant

Notebook này verify tất cả DoD items của TASK-09. Chạy từng cell theo thứ tự.

**Yêu cầu:** Neo4j và Qdrant đang chạy, file `.env` đã có credentials.

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

load_dotenv()

driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USER'), os.getenv('NEO4J_PASSWORD')),
)
qdrant = QdrantClient(
    host=os.getenv('QDRANT_HOST', 'localhost'),
    port=int(os.getenv('QDRANT_PORT', '6333')),
)
print('Kết nối thành công.')

## 1. Neo4j — Node Counts (DoD: 6 loại node đều có count > 0)

In [ ]:
with driver.session() as s:
    rows = s.run(
        'MATCH (n) RETURN labels(n)[0] as type, count(n) as count ORDER BY count DESC'
    ).data()

print(f"{'Node type':<15} {'Count':>8}")
print('-' * 25)
for r in rows:
    print(f"{r['type']:<15} {r['count']:>8}")

types_found = {r['type'] for r in rows}
required = {'Theme', 'Norm', 'Component', 'CTV', 'TextUnit', 'Jurisdiction'}
print(f"\n6 loại node đủ: {'✅' if required <= types_found else '❌ thiếu: ' + str(required - types_found)}")

## 2. Neo4j — [:IMPLEMENTS] chain (DoD: tier 2 → tier 1)

In [ ]:
with driver.session() as s:
    rows = s.run(
        'MATCH (n:Norm {tier:2})-[:IMPLEMENTS]->(p:Norm {tier:1}) '
        'RETURN n.id AS from_id, n.tier AS from_tier, p.id AS to_id, p.tier AS to_tier LIMIT 5'
    ).data()

print('[:IMPLEMENTS] chains (tier 2 → tier 1):')
for r in rows:
    print(f"  {r['from_id']} (tier {r['from_tier']}) → {r['to_id']} (tier {r['to_tier']})")
print(f"\nKết quả: {'✅ hợp lệ' if rows else '❌ không có chain'}")

## 3. Neo4j — [:APPLIES_TO] jurisdiction (DoD: jurisdiction đúng)

In [ ]:
with driver.session() as s:
    rows = s.run(
        'MATCH (n:Norm)-[:APPLIES_TO]->(j:Jurisdiction) '
        'RETURN n.id AS norm_id, j.name AS jurisdiction LIMIT 10'
    ).data()

print(f"{'norm_id':<45} {'jurisdiction'}")
print('-' * 60)
for r in rows:
    print(f"{r['norm_id']:<45} {r['jurisdiction']}")

## 4. Qdrant — Vector Counts (DoD: counts khớp với Neo4j)

In [ ]:
tu_count = qdrant.count(
    'legal_texts',
    count_filter=Filter(must=[FieldCondition(key='content_type', match=MatchValue(value='text_unit'))]),
).count
sm_count = qdrant.count(
    'legal_texts',
    count_filter=Filter(must=[FieldCondition(key='content_type', match=MatchValue(value='summary'))]),
).count

with driver.session() as s:
    neo4j_tu = s.run('MATCH (t:TextUnit) RETURN count(t) AS c').single()['c']
    neo4j_norm = s.run('MATCH (n:Norm) RETURN count(n) AS c').single()['c']

print(f"text_unit vectors : {tu_count:>5} | Neo4j TextUnit: {neo4j_tu:>5} | {'✅' if tu_count == neo4j_tu else '❌'}")
print(f"summary  vectors  : {sm_count:>5} | Neo4j Norm    : {neo4j_norm:>5} | {'✅' if sm_count == neo4j_norm else '❌'}")

## 5. Vector Search — Stage 1 Summary Routing

Query: `"phí chuyển mục đích sử dụng đất"` | Filter: `content_type="summary"`, `theme="dat-dai"`

In [ ]:
import sys
sys.path.insert(0, '..')
from src.ingestion.vectorizer import load_model, encode_text

model = load_model()

q1 = 'phí chuyển mục đích sử dụng đất'
vec1 = encode_text(model, q1)
res1 = qdrant.query_points(
    'legal_texts',
    query=vec1,
    limit=3,
    query_filter=Filter(must=[
        FieldCondition(key='content_type', match=MatchValue(value='summary')),
        FieldCondition(key='theme', match=MatchValue(value='dat-dai')),
    ]),
).points

print(f'Stage 1 query: "{q1}"')
print(f"{'Rank':<6} {'Score':<8} {'norm_id':<40} {'tier':<6} {'jurisdiction'}")
print('-' * 80)
for i, r in enumerate(res1, 1):
    print(f"{i:<6} {r.score:<8.4f} {r.payload['norm_id']:<40} {r.payload['tier']:<6} {r.payload['jurisdiction']}")

all_dat_dai = all(r.payload['theme'] == 'dat-dai' for r in res1)
print(f"\nTop-3 đều dat-dai: {'✅' if all_dat_dai else '❌'}")

## 6. Vector Search — Stage 2 Text Unit Retrieval

Query: `"đăng ký khai sinh"` | Filter: `content_type="text_unit"`, `jurisdiction="toan-quoc"`

> **Lưu ý:** Hiện tại chỉ có dữ liệu Đất đai ([A]). Kết quả sẽ trả về `dat-dai` cho đến khi [B] nộp file Hộ tịch và chạy lại pipeline ingestion.

In [ ]:
q2 = 'đăng ký khai sinh'
vec2 = encode_text(model, q2)
res2 = qdrant.query_points(
    'legal_texts',
    query=vec2,
    limit=3,
    query_filter=Filter(must=[
        FieldCondition(key='content_type', match=MatchValue(value='text_unit')),
        FieldCondition(key='jurisdiction', match=MatchValue(value='toan-quoc')),
    ]),
).points

print(f'Stage 2 query: "{q2}"')
print(f"{'Rank':<6} {'Score':<8} {'norm_id':<40} {'theme'}")
print('-' * 70)
for i, r in enumerate(res2, 1):
    print(f"{i:<6} {r.score:<8.4f} {r.payload['norm_id']:<40} {r.payload['theme']}")

print('\n⏳ Re-verify sau khi [B] nộp dữ liệu Hộ tịch.')

## 7. Idempotency Check

In [ ]:
with driver.session() as s:
    node_count = s.run('MATCH (n) RETURN count(n) AS c').single()['c']

total_vectors = qdrant.get_collection('legal_texts').points_count

print(f'Neo4j total nodes : {node_count}')
print(f'Qdrant total vectors: {total_vectors}')
print('\n(Chạy lại run_ingestion() + run_vectorization() rồi so sánh — số không được tăng.)')

In [ ]:
driver.close()
print('Done. Xem phase2_report.md để biết kết quả đầy đủ.')